# Your first Odin profile

Plot one L2 vertical profile and its stored measurement response. This template supports scalar lowercase L2 fields; older mixed-case or nested schemas require their own release reader. Uncertainty is optional and must match the chosen variable and units.

Download `selection.json` from Odin Data Access and save it next to this notebook. A planned release has no selectable files yet; do not substitute a migration directory or invented object URL. Read the selected release's schema, license and citation first.

Install the tested environment in a terminal: `python -m pip install duckdb==1.4.4 matplotlib==3.10.8 numpy==2.4.2 jupyterlab`. Restart the kernel after changing packages. Python 3.12 is tested.

The next cell downloads the **complete selected files**, up to a 64 MiB tutorial cap, verifies SHA256, and reuses a verified local cache. The website's date selection finds overlapping files; the files may contain observations outside those dates. No AWS account, keys or cloud query service is used. Exact bytes and rows depend on your selection; synthetic validation is described in the maintainer guide.

Keep `selection.json` and the generated provenance JSON with your result.


In [ ]:
import hashlib
import importlib.metadata
import json
import re
import sys
from pathlib import Path
from urllib.parse import urlsplit
from urllib.request import HTTPRedirectHandler, build_opener

import duckdb
import matplotlib.pyplot as plt
import numpy as np

# Tutorial safety limit, not an estimate of the whole dataset or query cost.
MAX_DOWNLOAD_BYTES = 64 * 1024 * 1024
CACHE = Path("odin-parquet-cache")
CACHE.mkdir(exist_ok=True)
TEMPLATE_VERSION = "odin-parquet-notebooks-1"
SELECTION_SOURCES = {}


class HTTPSRedirects(HTTPRedirectHandler):
    def redirect_request(self, request, fp, code, message, headers, new_url):
        target = urlsplit(new_url)
        if target.scheme != "https" or target.username or target.password:
            raise ValueError("Refusing a redirect outside anonymous HTTPS.")
        return super().redirect_request(request, fp, code, message, headers, new_url)


https_opener = build_opener(HTTPSRedirects())


def read_selection(filename):
    raw = Path(filename).read_bytes()
    selection = json.loads(raw)
    if selection.get("schema_version") != 1 or not selection.get("files"):
        raise ValueError("Use a non-empty version 1 selection exported by Odin Data Access.")
    for name in ("release", "dataset"):
        if not isinstance(selection.get(name), dict) or not selection[name].get("id"):
            raise ValueError(f"Missing {name} identity.")
    if not selection["release"].get("license") or not selection["release"].get("citation"):
        raise ValueError("The selection must retain its license and citation.")
    seen = set()
    for entry in selection["files"]:
        url = entry["url"]
        parsed = urlsplit(url)
        if (parsed.scheme != "https" or not parsed.hostname or parsed.username
                or parsed.password or parsed.query or parsed.fragment):
            raise ValueError("Expected permanent anonymous HTTPS object URLs.")
        if url in seen:
            raise ValueError("Duplicate object in selection.")
        seen.add(url)
        if not re.fullmatch(r"[0-9a-f]{64}", entry["sha256"]):
            raise ValueError("Missing lowercase SHA256 checksum.")
        for key in ("size_bytes", "row_count"):
            if type(entry[key]) is not int or entry[key] < 0:
                raise ValueError(f"Invalid {key}.")
    if sum(f["size_bytes"] for f in selection["files"]) > MAX_DOWNLOAD_BYTES:
        raise ValueError("Selection exceeds the tutorial cap; choose fewer files.")
    print("Release:", selection["release"]["id"], "Dataset:", selection["dataset"]["id"])
    print("Processing identity:", {k: selection["dataset"].get(k) for k in
          ("project", "product", "frequency_mode", "inversion_mode")})
    print("Declared files/bytes:", len(selection["files"]),
          sum(f["size_bytes"] for f in selection["files"]))
    print("Selection SHA256:", hashlib.sha256(raw).hexdigest())
    SELECTION_SOURCES[id(selection)] = {"filename": filename, "sha256": hashlib.sha256(raw).hexdigest()}
    return selection


def verified_files(selection):
    paths = []
    for entry in selection["files"]:
        destination = CACHE / (entry["sha256"] + ".parquet")
        if not destination.exists():
            temporary = destination.with_suffix(".part")
            try:
                count = 0
                with https_opener.open(entry["url"], timeout=60) as response, temporary.open("wb") as target:
                    if urlsplit(response.url).scheme != "https":
                        raise ValueError("HTTPS is required after redirects.")
                    while block := response.read(1024 * 1024):
                        count += len(block)
                        if count > entry["size_bytes"] or count > MAX_DOWNLOAD_BYTES:
                            raise ValueError("Download exceeded the declared byte count.")
                        target.write(block)
                with temporary.open("rb") as source:
                    digest = hashlib.file_digest(source, "sha256").hexdigest()
                if temporary.stat().st_size != entry["size_bytes"] or digest != entry["sha256"]:
                    raise ValueError("Downloaded bytes do not match the release checksum/size.")
                temporary.replace(destination)
            finally:
                temporary.unlink(missing_ok=True)
        with destination.open("rb") as source:
            digest = hashlib.file_digest(source, "sha256").hexdigest()
        if destination.stat().st_size != entry["size_bytes"] or digest != entry["sha256"]:
            raise ValueError("Cached file failed verification; inspect or remove that cached file.")
        paths.append(str(destination))
    return paths


def open_table(selection, table_name):
    paths = verified_files(selection)
    # Partition values remain in selection.json. No directory-based type inference.
    relation = con.read_parquet(paths, hive_partitioning=False, union_by_name=False)
    relation.create_view(table_name, replace=True)
    expected_rows = sum(f["row_count"] for f in selection["files"])
    actual_rows = con.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0]
    if actual_rows != expected_rows:
        raise ValueError("File row counts differ from the selected manifest.")
    return dict(con.execute(f'DESCRIBE "{table_name}"').fetchall()[i][:2]
                for i in range(len(relation.columns)))


def column_unit(selection, name):
    matches = [c for c in selection["dataset"]["columns"] if c["name"] == name]
    if len(matches) != 1 or not matches[0].get("unit"):
        raise ValueError(f"Read the release dictionary and supply a verified unit for {name}.")
    return matches[0]["unit"]


def require_columns(schema, *names):
    missing = set(names) - schema.keys()
    if missing:
        raise ValueError(f"This notebook does not support this physical schema; missing {sorted(missing)}.")


def provenance(selection, filename, analysis):
    receipt = {"template_version": TEMPLATE_VERSION, "analysis": analysis,
               "selection_source": SELECTION_SOURCES[id(selection)],
               "download_cap_bytes": MAX_DOWNLOAD_BYTES,
               "selection": selection, "python": sys.version,
               "packages": {p: importlib.metadata.version(p)
                            for p in ("duckdb", "matplotlib", "numpy")}}
    Path(filename).write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")


con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
selection = read_selection("selection.json")
schema = open_table(selection, "observations")
print("Physical columns:", schema)


## Run the example

Adjust the documented parameters below. The notebook stops for unsupported or ambiguous schemas instead of changing scientific identities or guessing units.

In [ ]:
if selection["dataset"]["level"] != "L2":
    raise ValueError("Choose a Level 2 vertical-profile dataset.")

# Select a supported variable after reading the selected release's dictionary.
VALUE_COLUMN = "vmr"             # For temperature retrievals, choose "temperature".
ERROR_COLUMN = None               # Optionally "error_noise" or "error_total".
if VALUE_COLUMN not in {"vmr", "temperature"} or ERROR_COLUMN not in {None, "error_noise", "error_total"}:
    raise ValueError("Unsupported variable/error choice.")
require_columns(schema, "scan_id", "altitude", VALUE_COLUMN, "meas_response")
if any("[]" in schema[k] or "STRUCT" in schema[k] for k in ("altitude", VALUE_COLUMN)):
    raise ValueError("This example expects scalar vertical-level rows, not profile arrays.")
value_unit = column_unit(selection, VALUE_COLUMN)
altitude_unit = column_unit(selection, "altitude")
# The smallest stored scan ID is a deterministic first example, not a time rule.
SCAN_ID = con.execute('SELECT min(scan_id) FROM observations').fetchone()[0]
if SCAN_ID is None:
    raise ValueError("No scan is present in these files.")
print("Example scan:", SCAN_ID)
fields = ["altitude", VALUE_COLUMN, "meas_response"]
if ERROR_COLUMN:
    require_columns(schema, ERROR_COLUMN)
    if column_unit(selection, ERROR_COLUMN) != value_unit:
        raise ValueError("The chosen uncertainty is not documented in the variable's units.")
    fields.append(ERROR_COLUMN)
rows = con.execute('SELECT ' + ', '.join('"' + x + '"' for x in fields)
                   + ' FROM observations WHERE scan_id = ? ORDER BY altitude', [SCAN_ID]).fetchall()
values = np.asarray(rows, dtype=float)  # null becomes NaN; never replace it by zero.
y, x, response = values[:, 0], values[:, 1], values[:, 2]
if len(np.unique(y[np.isfinite(y)])) != np.count_nonzero(np.isfinite(y)):
    raise ValueError("Repeated altitude for this scan: resolve product/version/row identity first.")
if not np.isfinite(x).any():
    raise ValueError("The selected variable is entirely missing; check the product and dictionary.")
fig, axes = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
axes[0].plot(x, y, "o-", label="Retrieved profile")
if ERROR_COLUMN:
    error = values[:, 3]
    if np.any(error[np.isfinite(error)] < 0):
        raise ValueError("Negative uncertainty values require scientific inspection.")
    axes[0].errorbar(x, y, xerr=error, fmt="none", label=ERROR_COLUMN)
axes[0].set(xlabel=f"{VALUE_COLUMN} [{value_unit}]", ylabel=f"Altitude [{altitude_unit}]")
axes[0].legend()
axes[1].plot(response, y, "o-")
axes[1].set_xlabel("Measurement response (see release definition)")
fig.suptitle(f"{selection['dataset']['product']} · scan {SCAN_ID}")
fig.tight_layout()
fig.savefig("odin-first-profile.png", dpi=150)
plt.show()
print("Rows / missing variable / missing response:", len(rows),
      np.count_nonzero(~np.isfinite(x)), np.count_nonzero(~np.isfinite(response)))
print("No quality or measurement-response threshold has been applied.")
provenance(selection, "first-profile-provenance.json", {
    "notebook": "first-profile", "scan_id": SCAN_ID, "value_column": VALUE_COLUMN,
    "error_column": ERROR_COLUMN, "vertical_coordinate": "altitude",
    "display_order": "altitude ascending", "quality_filter": None})


## Interpretation

These examples inspect stored data; they do not establish scientific validity or mission-wide completeness. Null values remain missing. Follow the release's product-specific quality and uncertainty guidance before drawing scientific conclusions. Preserve original coordinate/array order when using averaging kernels or other matrices; sorting a plotted profile does not reorder its kernel axes. Cite the release and processing project recorded in your selection. See [the analysis guide](https://odin-smr.org/data_access/parquet-guide) for formats and further analysis.